# 🧹 02 — Data Cleaning

Bu notebook, `data/raw/` klasöründeki 14 ham CSV dosyasını temizleyerek 
analiz için daha güvenilir ve tutarlı hale getirmeyi amaçlar.

### 🎯 Temizlik Aşamaları

- `\N` → `NaN` dönüşümü  
- Tarih kolonlarını datetime formatına çevirme  
- Sayısal kolonları numeric formatına dönüştürme  
- Gereksiz kolonları kaldırma (örn. `url`)  
- Yeni kolonlar oluşturma (örn. fullName)  
- Her dataset'i `data/processed/` altına **clean_*.csv** olarak kaydetme  

Temizlik, EDA aşamasında tespit edilen veri kalite problemlerine dayanmaktadır.


## 🧰 1. Ortam Kurulumu ve Temel Ayarlar

Bu bölümde:

- Gerekli Python kütüphanelerini yüklüyoruz  
- Ham (`data/raw/`) ve temizlenmiş (`data/processed/`) veri klasör yollarını tanımlıyoruz  
- Kullanacağımız CSV dosya isimlerini merkezi bir listede topluyoruz  


In [1]:
import os
import pandas as pd
import numpy as np

RAW_DATA_PATH = "../data/raw/"
PROCESSED_DATA_PATH = "../data/processed/"

CSV_FILES = [
    "Constructor_Performance.csv",
    "Constructor_Rankings.csv",
    "Driver_Details.csv",
    "Driver_Rankings.csv",
    "Lap_Timings.csv",
    "Pit_Stop_Records.csv",
    "Qualifying_Results.csv",
    "Race_Results.csv",
    "Race_Schedule.csv",
    "Race_Status.csv",
    "Season_Summaries.csv",
    "Sprint_Race_Results.csv",
    "Team_Details.csv",
    "Track_Information.csv",
]

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

print("Raw files:", os.listdir(RAW_DATA_PATH))
print("Processed folder ready:", PROCESSED_DATA_PATH)


Raw files: ['Sprint_Race_Results.csv', 'Qualifying_Results.csv', 'Track_Information.csv', 'Constructor_Rankings.csv', 'Constructor_Performance.csv', 'Race_Status.csv', 'Driver_Rankings.csv', 'Race_Schedule.csv', 'Team_Details.csv', 'Lap_Timings.csv', 'Driver_Details.csv', 'Race_Results.csv', 'Pit_Stop_Records.csv', 'Season_Summaries.csv']
Processed folder ready: ../data/processed/


## 🧽 2. Genel Temizlik Yardımcı Fonksiyonları

Bu bölümde tekrar eden temizlik adımlarını fonksiyonlara dönüştürüyoruz:

- `\N` → `NaN` dönüşümü  
- Belirli kolonları numeric tipine çevirme  
- Belirli kolonları datetime tipine çevirme  

Bu sayede her veri seti için yazacağımız temizlik kodu daha kısa ve okunabilir olacak.


In [2]:
def load_raw_csv(filename: str) -> pd.DataFrame:
    path = os.path.join(RAW_DATA_PATH, filename)
    df = pd.read_csv(path)
    df = df.replace("\\N", np.nan)
    return df


def to_numeric(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Verilen kolonları sayısal tipe çevirir (hata durumunda NaN).
    """
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def to_datetime(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Verilen kolonları datetime tipine çevirir (hata durumunda NaT).
    """
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


## 🧑‍✈️ 3. Driver_Details → clean_drivers.csv

Bu adımda sürücü bilgilerini temizleyerek `clean_drivers.csv` oluşturuyoruz.

Yapılanlar:
- `\N` → `NaN`
- `dob` kolonunu datetime formatına çevirme
- `number` kolonunu numeric formata çevirme
- `fullName = forename + ' ' + surname` kolonu ekleme
- Tüm temizlenmiş veriyi `data/processed/clean_drivers.csv` olarak kaydetme


In [3]:
drivers = load_raw_csv("Driver_Details.csv")

drivers = to_datetime(drivers, ["dob"])
drivers = to_numeric(drivers, ["number"])

drivers["fullName"] = drivers["forename"] + " " + drivers["surname"]

drivers.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_drivers.csv"), index=False)
print("✅ Saved clean_drivers.csv")
drivers.head()


✅ Saved clean_drivers.csv


,driverId,driverRef,number,code,forename,surname,dob,nationality,url,fullName
0,1,hamilton,44.0,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton,Lewis Hamilton
1,2,heidfeld,NaN,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld,Nick Heidfeld
2,3,rosberg,6.0,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg,Nico Rosberg
3,4,alonso,14.0,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso,Fernando Alonso
4,5,kovalainen,NaN,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen,Heikki Kovalainen


## 🏁 4. Team_Details → clean_constructors.csv

Bu adımda takım (constructor) bilgilerini temizleyerek `clean_constructors.csv` oluşturuyoruz.

Yapılanlar:
- `\N` → `NaN`
- Gereksiz `url` kolonunu kaldırma
- Tüm temizlenmiş veriyi `data/processed/clean_constructors.csv` olarak kaydetme


In [4]:
constructors = load_raw_csv("Team_Details.csv")

# URL kolonuna analizde ihtiyaç yoksa kaldırabiliriz
if "url" in constructors.columns:
    constructors = constructors.drop(columns=["url"])

constructors.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_constructors.csv"), index=False)
print("✅ Saved clean_constructors.csv")
constructors.head()


✅ Saved clean_constructors.csv


,constructorId,constructorRef,name,nationality
0,1,mclaren,McLaren,British
1,2,bmw_sauber,BMW Sauber,German
2,3,williams,Williams,British
3,4,renault,Renault,French
4,5,toro_rosso,Toro Rosso,Italian


## 🏟 5. Track_Information → clean_circuits.csv

Bu adımda pist (circuit) bilgilerini temizleyerek `clean_circuits.csv` oluşturuyoruz.

Yapılanlar:
- `\N` → `NaN`
- Coğrafi koordinatları numeric formata çevirme (`lat`, `lng`, `alt`)
- Gereksiz `url` kolonunu kaldırma


In [5]:
circuits = load_raw_csv("Track_Information.csv")

circuits = to_numeric(circuits, ["lat", "lng", "alt"])

if "url" in circuits.columns:
    circuits = circuits.drop(columns=["url"])

circuits.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_circuits.csv"), index=False)
print("✅ Saved clean_circuits.csv")
circuits.head()


✅ Saved clean_circuits.csv


,circuitId,circuitRef,name,location,country,lat,lng,alt
0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.84970,144.96800,10
1,2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.73800,18
2,3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.03250,50.51060,7
3,4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57000,2.26111,109
4,5,istanbul,Istanbul Park,Istanbul,Turkey,40.95170,29.40500,130


## 🗓 6. Race_Schedule & Season_Summaries → clean_races.csv, clean_seasons.csv

Bu adımda:

- Yarış takvimi bilgisini `clean_races.csv`
- Sezon özetlerini `clean_seasons.csv`

olarak temizleyip kaydediyoruz.

Yapılanlar:
- `\N` → `NaN`
- `date` ve `time` gibi kolonları datetime/time formatına çevirmek (en azından `date`)
- Season_Summaries için basit temizlik ve yeniden adlandırma


In [6]:
# Race schedule
race_schedule = load_raw_csv("Race_Schedule.csv")
race_schedule = to_datetime(race_schedule, ["date", "fp1_date", "fp2_date", "fp3_date", "quali_date", "sprint_date"])

race_schedule.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_races.csv"), index=False)
print("✅ Saved clean_races.csv")

# Season summaries
seasons = load_raw_csv("Season_Summaries.csv")
seasons = to_numeric(seasons, ["rounds"])

seasons.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_seasons.csv"), index=False)
print("✅ Saved clean_seasons.csv")

race_schedule.head(), seasons.head()


✅ Saved clean_races.csv
✅ Saved clean_seasons.csv


(   raceId  year  round  circuitId                   name       date      time  \
 0       1  2009      1          1  Australian Grand Prix 2009-03-29  06:00:00   
 1       2  2009      2          2   Malaysian Grand Prix 2009-04-05  09:00:00   
 2       3  2009      3         17     Chinese Grand Prix 2009-04-19  07:00:00   
 3       4  2009      4          3     Bahrain Grand Prix 2009-04-26  12:00:00   
 4       5  2009      5          4     Spanish Grand Prix 2009-05-10  12:00:00   
 
                                                  url fp1_date fp1_time  \
 0  http://en.wikipedia.org/wiki/2009_Australian_G...      NaT      NaN   
 1  http://en.wikipedia.org/wiki/2009_Malaysian_Gr...      NaT      NaN   
 2  http://en.wikipedia.org/wiki/2009_Chinese_Gran...      NaT      NaN   
 3  http://en.wikipedia.org/wiki/2009_Bahrain_Gran...      NaT      NaN   
 4  http://en.wikipedia.org/wiki/2009_Spanish_Gran...      NaT      NaN   
 
   fp2_date fp2_time fp3_date fp3_time quali_date qual

## 📋 7. Race_Status → clean_status.csv

Bu adımda yarış sonucu durum açıklamalarını içeren tabloyu 
`clean_status.csv` olarak kaydediyoruz.

Bu tablo daha sonra `Race_Results` ve `Sprint_Race_Results` ile join edilerek
DNF/finish nedenlerini anlamak için kullanılacaktır.


In [7]:
status_df = load_raw_csv("Race_Status.csv")

status_df.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_status.csv"), index=False)
print("✅ Saved clean_status.csv")
status_df.head()


✅ Saved clean_status.csv


,statusId,status
0,1,Finished
1,2,Disqualified
2,3,Accident
3,4,Collision
4,5,Engine


## 🏎 8. Race_Results → clean_race_results.csv

Bu adımda ana yarış sonuçlarını temizleyerek `clean_race_results.csv` oluşturuyoruz.

Yapılanlar:
- `\N` → `NaN`
- Sayısal kolonları numeric formata çevirme (grid, position, points, laps, milliseconds, fastestLapSpeed vs.)
- `statusId` üzerinden `clean_status.csv` ile join edip `status` açıklamasını ekleme


In [8]:
race_results = load_raw_csv("Race_Results.csv")

numeric_cols = [
    "number", "grid", "position", "positionOrder",
    "points", "laps", "milliseconds", "fastestLap",
    "rank", "fastestLapSpeed", "statusId"
]
race_results = to_numeric(race_results, numeric_cols)

clean_status = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, "clean_status.csv"))

race_results = race_results.merge(clean_status, how="left", on="statusId")

race_results.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_race_results.csv"), index=False)
print("✅ Saved clean_race_results.csv")
race_results.head()


✅ Saved clean_race_results.csv


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,status
0,1,18,1,1,22.0,1,1.0,1,1,10.0,58,1:34:50.616,5690616.0,39.0,2.0,1:27.452,218.300,1,Finished
1,2,18,2,2,3.0,5,2.0,2,2,8.0,58,+5.478,5696094.0,41.0,3.0,1:27.739,217.586,1,Finished
2,3,18,3,3,7.0,7,3.0,3,3,6.0,58,+8.163,5698779.0,41.0,5.0,1:28.090,216.719,1,Finished
3,4,18,4,4,5.0,11,4.0,4,4,5.0,58,+17.181,5707797.0,58.0,7.0,1:28.603,215.464,1,Finished
4,5,18,5,1,23.0,3,5.0,5,5,4.0,58,+18.014,5708630.0,43.0,1.0,1:27.418,218.385,1,Finished


## ⚡ 9. Sprint_Race_Results → clean_sprint_results.csv

Bu adımda sprint yarış sonuçlarını temizleyerek `clean_sprint_results.csv` oluşturuyoruz.

Yapılanlar:
- `\N` → `NaN`
- Sayısal kolonları numeric formata çevirme
- `statusId` üzerinden `clean_status.csv` ile join edip `status` açıklamasını ekleme


In [9]:
sprint_results = load_raw_csv("Sprint_Race_Results.csv")

numeric_cols = [
    "number", "grid", "position", "positionOrder",
    "points", "laps", "milliseconds", "fastestLap",
    "rank", "fastestLapSpeed", "statusId"
]
sprint_results = to_numeric(sprint_results, numeric_cols)

sprint_results = sprint_results.merge(clean_status, how="left", on="statusId")

sprint_results.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_sprint_results.csv"), index=False)
print("✅ Saved clean_sprint_results.csv")
sprint_results.head()


✅ Saved clean_sprint_results.csv


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId,status
0,1,1061,830,9,33,2,1.0,1,1,3,17,25:38.426,1538426.0,14.0,1:30.013,1,Finished
1,2,1061,1,131,44,1,2.0,2,2,2,17,+1.430,1539856.0,17.0,1:29.937,1,Finished
2,3,1061,822,131,77,3,3.0,3,3,1,17,+7.502,1545928.0,17.0,1:29.958,1,Finished
3,4,1061,844,6,16,4,4.0,4,4,0,17,+11.278,1549704.0,16.0,1:30.163,1,Finished
4,5,1061,846,1,4,6,5.0,5,5,0,17,+24.111,1562537.0,16.0,1:30.566,1,Finished


## 📊 10. Standings & Constructor Performance → clean_driver_standings.csv, clean_constructor_standings.csv, clean_constructor_results.csv

Bu adımda sezon içi sıralamaları ve takım performansını içeren tabloları temizliyoruz.

Yapılanlar:
- `\N` → `NaN`
- Puan ve pozisyon kolonlarını numeric formata çevirme
- Temizlenmiş tabloları ayrı ayrı kaydetme


In [10]:
# Driver standings
driver_standings = load_raw_csv("Driver_Rankings.csv")
driver_standings = to_numeric(driver_standings, ["raceId", "driverId", "points", "position", "wins"])

driver_standings.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_driver_standings.csv"), index=False)
print("✅ Saved clean_driver_standings.csv")

# Constructor standings
constructor_standings = load_raw_csv("Constructor_Rankings.csv")
constructor_standings = to_numeric(constructor_standings, ["raceId", "constructorId", "points", "position", "wins"])

constructor_standings.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_constructor_standings.csv"), index=False)
print("✅ Saved clean_constructor_standings.csv")

# Constructor performance
constructor_perf = load_raw_csv("Constructor_Performance.csv")
constructor_perf = to_numeric(constructor_perf, ["raceId", "constructorId", "points"])

constructor_perf.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_constructor_results.csv"), index=False)
print("✅ Saved clean_constructor_results.csv")

driver_standings.head(), constructor_standings.head(), constructor_perf.head()


✅ Saved clean_driver_standings.csv
✅ Saved clean_constructor_standings.csv
✅ Saved clean_constructor_results.csv


(   driverStandingsId  raceId  driverId  points  position positionText  wins
 0                  1      18         1    10.0         1            1     1
 1                  2      18         2     8.0         2            2     0
 2                  3      18         3     6.0         3            3     0
 3                  4      18         4     5.0         4            4     0
 4                  5      18         5     4.0         5            5     0,
    constructorStandingsId  raceId  constructorId  points  position  \
 0                       1      18              1    14.0         1   
 1                       2      18              2     8.0         3   
 2                       3      18              3     9.0         2   
 3                       4      18              4     5.0         4   
 4                       5      18              5     2.0         5   
 
   positionText  wins  
 0            1     1  
 1            3     0  
 2            2     0  
 3           

## ⏱ 11. Lap_Timings → clean_lap_times.csv

Bu adımda tur zamanlarını içeren tabloyu temizliyoruz.

Şimdilik:
- `\N` → `NaN`
- `raceId`, `driverId`, `lap`, `position` kolonlarını numeric formata çevirme
- `time` kolonunu string olarak bırakma (ileride isteğe bağlı parse edilebilir)


In [11]:
lap_times = load_raw_csv("Lap_Timings.csv")

lap_times = to_numeric(lap_times, ["raceId", "driverId", "lap", "position"])

lap_times.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_lap_times.csv"), index=False)
print("✅ Saved clean_lap_times.csv")
lap_times.head()


✅ Saved clean_lap_times.csv


,raceId,driverId,lap,position,time,milliseconds
0,841,20,1,1,1:38.109,98109
1,841,20,2,1,1:33.006,93006
2,841,20,3,1,1:32.713,92713
3,841,20,4,1,1:32.803,92803
4,841,20,5,1,1:32.342,92342


## 🛑 12. Pit_Stop_Records → clean_pit_stops.csv

Bu adımda pit stop verilerini temizliyoruz.

Yapılanlar:
- `\N` → `NaN`
- `raceId`, `driverId`, `stop`, `lap`, `milliseconds` kolonlarını numeric formata çevirme
- `duration` ve `time` kolonlarını string olarak bırakma (isteğe bağlı parse edilebilir)


In [12]:
pit_stops = load_raw_csv("Pit_Stop_Records.csv")

pit_stops = to_numeric(pit_stops, ["raceId", "driverId", "stop", "lap", "milliseconds"])

pit_stops.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_pit_stops.csv"), index=False)
print("✅ Saved clean_pit_stops.csv")
pit_stops.head()


✅ Saved clean_pit_stops.csv


,raceId,driverId,stop,lap,time,duration,milliseconds
0,841,153,1,1,17:05:23,26.898,26898
1,841,30,1,1,17:05:52,25.021,25021
2,841,17,1,11,17:20:48,23.426,23426
3,841,4,1,12,17:22:34,23.251,23251
4,841,13,1,13,17:24:10,23.842,23842


## 🏁 13. Qualifying_Results → clean_qualifying_results.csv

Bu adımda sıralama turlarını içeren tabloyu temizliyoruz.

Yapılanlar:
- `\N` → `NaN`
- `raceId`, `driverId`, `constructorId`, `number`, `position` kolonlarını numeric formata çevirme
- `q1`, `q2`, `q3` kolonlarını şimdilik string formatında bırakma (ileride süreye dönüştürülebilir)


In [13]:
qualifying = load_raw_csv("Qualifying_Results.csv")

qualifying = to_numeric(qualifying, ["raceId", "driverId", "constructorId", "number", "position"])

qualifying.to_csv(os.path.join(PROCESSED_DATA_PATH, "clean_qualifying_results.csv"), index=False)
print("✅ Saved clean_qualifying_results.csv")
qualifying.head()


✅ Saved clean_qualifying_results.csv


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
0,1,18,1,1,22,1,1:26.572,1:25.187,1:26.714
1,2,18,9,2,4,2,1:26.103,1:25.315,1:26.869
2,3,18,5,1,23,3,1:25.664,1:25.452,1:27.079
3,4,18,13,6,2,4,1:25.994,1:25.691,1:27.178
4,5,18,2,2,3,5,1:25.960,1:25.518,1:27.236


# ✅ 14. Genel Temizlik Özeti

Bu notebook'ta:

- `data/raw/` altındaki 14 ham CSV dosyası yüklendi
- `\N` değerleri sistematik olarak `NaN` ile değiştirildi
- Tarih ve sayısal kolonlar uygun veri tiplerine dönüştürüldü
- Gereksiz gördüğümüz bazı kolonlar (örn. `url`) kaldırıldı
- Bazı veri setlerinde ek kolonlar oluşturuldu (örn. `fullName`)
- Her veri seti `data/processed/` altında **clean_*.csv** olarak kaydedildi

Artık:

- Keşifsel analiz (EDA)  
- Dashboard oluşturma  
- İleri düzey modelleme  

için kullanılabilecek **temiz ve tutarlı** bir veri yapısına sahibiz.
